In [1]:
import os
import json
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
FINE_TUNING_DIR   = Path("/kaggle/input/notebooks/adityarajpaul/04-fine-tuning/fine_tuning")
PREPROCESSING_DIR = Path("/kaggle/input/notebooks/adityarajpaul/02-preprocessing/exports")
OUTPUT_DIR        = Path("/kaggle/working/export")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = FINE_TUNING_DIR / "finetuned_best.keras"

print("TensorFlow version:", tf.__version__)
print("Model path exists:", MODEL_PATH.exists())
print("class_mapping exists:", (PREPROCESSING_DIR / "class_mapping.json").exists())

2026-06-09 13:57:16.589782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781013436.873225      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781013436.953163      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781013437.604315      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781013437.604357      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781013437.604362      16 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
Model path exists: True
class_mapping exists: True


In [2]:
with open(PREPROCESSING_DIR / "class_mapping.json") as f:
    class_mapping = json.load(f)   # {"0": "Apple___Apple_scab", ...}

# ── Clean display name helper ──────────────────────────────────────────────────
def clean_label(raw):
    parts   = raw.split("___")
    plant   = parts[0].replace("_", " ")
    disease = parts[1].replace("_", " ").replace("(", "").replace(")", "") if len(parts) > 1 else ""
    return f"{plant}: {disease}".strip(": ")

idx_to_class   = {int(k): v for k, v in class_mapping.items()}
idx_to_display = {k: clean_label(v) for k, v in idx_to_class.items()}

# ── Save clean mapping as JSON for backend ─────────────────────────────────────
clean_mapping = {str(k): v for k, v in idx_to_display.items()}
with open(OUTPUT_DIR / "disease_labels.json", "w") as f:
    json.dump(clean_mapping, f, indent=2)

# ── Generate disease_labels.py for direct import in backend ───────────────────
py_lines = [
    "# Auto-generated by 06_export.ipynb — do not edit manually",
    "# Maps model output index (int) to human-readable disease name",
    "",
    "DISEASE_LABELS = {",
]
for idx in sorted(idx_to_display.keys()):
    py_lines.append(f'    {idx}: "{idx_to_display[idx]}",')
py_lines.append("}")
py_lines.append("")
py_lines.append("NUM_CLASSES = 38")
py_lines.append("")
py_lines.append("def get_label(idx: int) -> str:")
py_lines.append('    """Return display name for a predicted class index."""')
py_lines.append("    return DISEASE_LABELS.get(idx, f\"Unknown class {idx}\")")
py_lines.append("")

py_content = "\n".join(py_lines)
with open(OUTPUT_DIR / "disease_labels.py", "w") as f:
    f.write(py_content)

print("disease_labels.json saved.")
print("disease_labels.py saved.")
print(f"\nSample entries:")
for i in [0, 1, 2, 37]:
    print(f"  {i}: {idx_to_display[i]}")

disease_labels.json saved.
disease_labels.py saved.

Sample entries:
  0: Apple: Apple scab
  1: Apple: Black rot
  2: Apple: Cedar apple rust
  37: Tomato: healthy


In [3]:
@tf.keras.utils.register_keras_serializable()
def top3_accuracy(y_true, y_pred):
    return tf.keras.metrics.sparse_top_k_categorical_accuracy(y_true, y_pred, k=3)

print("Loading model...")
model = keras.models.load_model(
    str(MODEL_PATH),
    custom_objects={"top3_accuracy": top3_accuracy}
)
print("Model loaded successfully.")
model.summary(show_trainable=False, expand_nested=False)

Loading model...


2026-06-09 13:57:36.851545: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model loaded successfully.


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 112 variables whereas the saved optimizer has 222 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1536)           │         6,144 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,992,699 (76.27 MB)

 Trainable params: 8,799,780 (33.57 MB)

 Non-trainable params: 2,393,137 (9.13 MB)

 Optimizer params: 8,799,782 (33.57 MB)

In [4]:
SAVEDMODEL_PATH = OUTPUT_DIR / "crop_disease_model"

print(f"Exporting to SavedModel format at: {SAVEDMODEL_PATH}")
model.export(str(SAVEDMODEL_PATH))

# Verify the export produced the expected directory structure
print("\nExported files:")
for f in sorted(SAVEDMODEL_PATH.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {str(f.relative_to(SAVEDMODEL_PATH)):<50} {size_mb:.2f} MB")

Exporting to SavedModel format at: /kaggle/working/export/crop_disease_model
INFO:tensorflow:Assets written to: /kaggle/working/export/crop_disease_model/assets


INFO:tensorflow:Assets written to: /kaggle/working/export/crop_disease_model/assets


Saved artifact at '/kaggle/working/export/crop_disease_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 300, 300, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  139596784649040: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139596784651920: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  139596934471248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934468368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934472016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934471632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934472400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934471824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934472208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139596934471056: TensorSpec(shape=(), d

In [5]:
# Generate a synthetic test image (random pixels, valid shape)
# This is sufficient to verify the export — we're checking shape and output validity,
# not prediction correctness
print("Generating synthetic test image for sanity check...")
synthetic_image = tf.random.uniform(shape=(1, 300, 300, 3), minval=0, maxval=255, dtype=tf.float32)

# Load the exported SavedModel fresh (simulates how FastAPI will load it)
print("Loading exported SavedModel...")
loaded_model = tf.saved_model.load(str(SAVEDMODEL_PATH))
infer = loaded_model.signatures["serving_default"]
print("SavedModel loaded via tf.saved_model.load()")

# Run inference
output     = infer(synthetic_image)
output_key = list(output.keys())[0]
probs      = output[output_key].numpy()[0]   # shape: (38,)

top3_idx   = np.argsort(probs)[-3:][::-1]
top3_probs = probs[top3_idx]

print(f"\nInput shape:          {synthetic_image.shape}")
print(f"Output shape:         {probs.shape}")
print(f"Probabilities sum to: {probs.sum():.6f}")
print(f"Output key name:      '{output_key}'")
print(f"\nTop-3 predictions (on random input — labels meaningless):")
for rank, (idx, prob) in enumerate(zip(top3_idx, top3_probs)):
    print(f"  {rank+1}. [{idx}] {idx_to_display[idx]:<45} {prob:.4f}")

assert probs.shape == (38,),          f"Expected shape (38,), got {probs.shape}"
assert abs(probs.sum() - 1.0) < 1e-4, f"Probabilities don't sum to 1: {probs.sum()}"
assert output_key is not None,         "No output key found in signatures"

print("\nSanity check passed — export is valid.")

Generating synthetic test image for sanity check...
Loading exported SavedModel...
SavedModel loaded via tf.saved_model.load()

Input shape:          (1, 300, 300, 3)
Output shape:         (38,)
Probabilities sum to: 1.000000
Output key name:      'output_0'

Top-3 predictions (on random input — labels meaningless):
  1. [0] Apple: Apple scab                             0.7299
  2. [30] Tomato: Late blight                           0.1237
  3. [24] Soybean: healthy                              0.0605

Sanity check passed — export is valid.


In [6]:
# Calculate total export size
total_size = sum(
    f.stat().st_size for f in SAVEDMODEL_PATH.rglob("*") if f.is_file()
) / (1024 * 1024)

print("=" * 60)
print("EXPORT SUMMARY")
print("=" * 60)
print(f"SavedModel directory:  {SAVEDMODEL_PATH.name}/")
print(f"Total export size:     {total_size:.2f} MB")
print(f"Labels JSON:           disease_labels.json")
print(f"Labels Python:         disease_labels.py")
print()
print("Files saved to /kaggle/working/export/:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_dir():
        dir_size = sum(x.stat().st_size for x in f.rglob("*") if x.is_file()) / (1024*1024)
        print(f"  {f.name}/ (directory)  {dir_size:.2f} MB")
    else:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name:<45} {size_mb:.2f} MB")
print()
print("Phase 1 complete. Next steps:")
print("  1. Commit this notebook")
print("  2. Download crop_disease_model/ → place in backend/ml_models/")
print("  3. Copy disease_labels.py → backend/app/utils/disease_labels.py")
print("  4. Proceed to Phase 2 — FastAPI backend")

EXPORT SUMMARY
SavedModel directory:  crop_disease_model/
Total export size:     89.16 MB
Labels JSON:           disease_labels.json
Labels Python:         disease_labels.py

Files saved to /kaggle/working/export/:
  crop_disease_model/ (directory)  89.16 MB
  disease_labels.json                           0.00 MB
  disease_labels.py                             0.00 MB

Phase 1 complete. Next steps:
  1. Commit this notebook
  2. Download crop_disease_model/ → place in backend/ml_models/
  3. Copy disease_labels.py → backend/app/utils/disease_labels.py
  4. Proceed to Phase 2 — FastAPI backend
